<a href="https://colab.research.google.com/github/skrixh/enterprise-retail-data-platform/blob/main/python/26MAS1015/ERDP_Data_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load 5 cleaned datasets

In [60]:
import pandas as pd

pos_clean = pd.read_csv("/content/pos_transactions_clean.csv")
ecommerce_clean = pd.read_csv("/content/ecommerce_orders_clean.csv")
crm_clean = pd.read_csv("/content/customers_clean.csv")
inventory_clean = pd.read_csv("/content/inventory_clean.csv")
supplier_clean = pd.read_csv("/content/supplier_orders_clean.csv")

Display shapes

In [61]:
print("\nPOS:", pos_clean.shape)
print("E-Commerce:", ecommerce_clean.shape)
print("CRM:", crm_clean.shape)
print("Inventory:", inventory_clean.shape)
print("Supplier:", supplier_clean.shape)


POS: (10000, 8)
E-Commerce: (5000, 8)
CRM: (3000, 6)
Inventory: (3000, 6)
Supplier: (2000, 8)


Initialize validation

In [62]:
#print("DATA VALIDATION - ERDP")

import pandas as pd

validation_results = []

Duplicate validation

In [63]:
#print("1. DUPLICATE VALIDATION")

datasets = {
    "POS": pos_clean,
    "E-Commerce": ecommerce_clean,
    "CRM": crm_clean,
    "Inventory": inventory_clean,
    "Supplier": supplier_clean
}

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    validation_results.append([
        name,
        "Duplicate Rows",
        duplicate_count,
        "PASS" if duplicate_count == 0 else "FAIL"
    ])


Missing value validation

In [64]:
#print("2. MISSING VALUE VALIDATION")

for name, df in datasets.items():

    missing_count = df.isnull().sum().sum()

    validation_results.append([
        name,
        "Missing Values",
        missing_count,
        "PASS" if missing_count == 0 else "FAIL"
    ])



Customer ID validation

In [65]:
# print("3. CUSTOMER ID VALIDATION")
# print("POS → CRM")
# print("E-Commerce → CRM")

crm_customers = set(
    crm_clean["customer_id"].dropna().astype(str)
)

pos_customers = set(
    pos_clean.loc[
        pos_clean["customer_id"] != "UNKNOWN",
        "customer_id"
    ].dropna().astype(str)
)

missing_pos_customers = pos_customers - crm_customers

validation_results.append([
    "POS",
    "Customer IDs not found in CRM",
    len(missing_pos_customers),
    "PASS" if len(missing_pos_customers) == 0 else "FAIL"
])


ecommerce_customers = set(
    ecommerce_clean.loc[
        ecommerce_clean["customer_id"] != "UNKNOWN",
        "customer_id"
    ].dropna().astype(str)
)

missing_ecommerce_customers = ecommerce_customers - crm_customers

validation_results.append([
    "E-Commerce",
    "Customer IDs not found in CRM",
    len(missing_ecommerce_customers),
    "PASS" if len(missing_ecommerce_customers) == 0 else "FAIL"
])


Product ID validation

In [66]:
# print("POS → INVENTORY")
# print("E-Commerce → INVENTORY")
# print("Supplier → INVENTORY")

inventory_products = set(
    inventory_clean["product_id"].dropna().astype(str)
)


pos_products = set(
    pos_clean.loc[
        pos_clean["product_id"] != "UNKNOWN",
        "product_id"
    ].dropna().astype(str)
)

missing_pos_products = pos_products - inventory_products

validation_results.append([
    "POS",
    "Product IDs not found in Inventory",
    len(missing_pos_products),
    "PASS" if len(missing_pos_products) == 0 else "FAIL"
])


ecommerce_products = set(
    ecommerce_clean.loc[
        ecommerce_clean["product_id"] != "UNKNOWN",
        "product_id"
    ].dropna().astype(str)
)

missing_ecommerce_products = ecommerce_products - inventory_products

validation_results.append([
    "E-Commerce",
    "Product IDs not found in Inventory",
    len(missing_ecommerce_products),
    "PASS" if len(missing_ecommerce_products) == 0 else "FAIL"
])


supplier_products = set(
    supplier_clean.loc[
        supplier_clean["product_id"] != "UNKNOWN",
        "product_id"
    ].dropna().astype(str)
)

missing_supplier_products = supplier_products - inventory_products

validation_results.append([
    "Supplier",
    "Product IDs not found in Inventory",
    len(missing_supplier_products),
    "PASS" if len(missing_supplier_products) == 0 else "FAIL"
])


Negative value validation

In [67]:
# print("5. NEGATIVE VALUE VALIDATION")

checks = [
    ("POS", "Negative Quantity", (pos_clean["quantity"] < 0).sum()),
    ("POS", "Negative Unit Price", (pos_clean["unit_price"] < 0).sum()),

    ("E-Commerce", "Negative Quantity", (ecommerce_clean["quantity"] < 0).sum()),
    ("E-Commerce", "Negative Unit Price", (ecommerce_clean["unit_price"] < 0).sum()),

    ("Inventory", "Negative Stock Quantity", (inventory_clean["stock_quantity"] < 0).sum()),
    ("Inventory", "Negative Reorder Level", (inventory_clean["reorder_level"] < 0).sum()),

    ("Supplier", "Negative Quantity", (supplier_clean["quantity"] < 0).sum()),
    ("Supplier", "Negative Unit Cost", (supplier_clean["unit_cost"] < 0).sum())
]

for dataset, check, count in checks:

    validation_results.append([
        dataset,
        check,
        count,
        "PASS" if count == 0 else "FAIL"
    ])


Date validation

In [68]:
# print("6. DATE VALIDATION")

# Convert date columns to datetime objects
pos_clean["transaction_date"] = pd.to_datetime(pos_clean["transaction_date"])
ecommerce_clean["order_date"] = pd.to_datetime(ecommerce_clean["order_date"])
supplier_clean["order_date"] = pd.to_datetime(supplier_clean["order_date"])
supplier_clean["expected_delivery_date"] = pd.to_datetime(supplier_clean["expected_delivery_date"])

# POS transaction date should not be in the future
future_pos = (
    pos_clean["transaction_date"] > pd.Timestamp.today()
).sum()

validation_results.append([
    "POS",
    "Future Transaction Dates",
    future_pos,
    "PASS" if future_pos == 0 else "FAIL"
])


# E-Commerce order date
future_ecommerce = (
    ecommerce_clean["order_date"] > pd.Timestamp.today()
).sum()

validation_results.append([
    "E-Commerce",
    "Future Order Dates",
    future_ecommerce,
    "PASS" if future_ecommerce == 0 else "FAIL"
])


# Supplier delivery date should not be before order date
invalid_delivery = (
    supplier_clean["expected_delivery_date"]
    < supplier_clean["order_date"]
).sum()

validation_results.append([
    "Supplier",
    "Delivery Date Before Order Date",
    invalid_delivery,
    "PASS" if invalid_delivery == 0 else "FAIL"
])

Primary key validation

In [69]:
# print("7. PRIMARY KEY VALIDATION")

checks = [
    ("CRM", "Duplicate Customer IDs",
     crm_clean["customer_id"].duplicated().sum()),

    ("Inventory", "Duplicate Inventory IDs",
     inventory_clean["inventory_id"].duplicated().sum()),

    ("Supplier", "Duplicate Purchase Order IDs",
     supplier_clean["purchase_order_id"].duplicated().sum())
]

for dataset, check, count in checks:

    validation_results.append([
        dataset,
        check,
        count,
        "PASS" if count == 0 else "FAIL"
    ])



Final validation summary

In [70]:
# print("FINAL VALIDATION TABLE")

validation_summary = pd.DataFrame(
    validation_results,
    columns=[
        "Dataset",
        "Validation_Check",
        "Issue_Count",
        "Status"
    ]
)

print("========== DATA VALIDATION SUMMARY ==========")

display(validation_summary)

========== DATA VALIDATION SUMMARY ==========


,Dataset,Validation_Check,Issue_Count,Status
0,POS,Duplicate Rows,0,PASS
1,E-Commerce,Duplicate Rows,0,PASS
2,CRM,Duplicate Rows,0,PASS
3,Inventory,Duplicate Rows,0,PASS
4,Supplier,Duplicate Rows,0,PASS
5,POS,Missing Values,0,PASS
6,E-Commerce,Missing Values,0,PASS
7,CRM,Missing Values,0,PASS
8,Inventory,Missing Values,0,PASS
9,Supplier,Missing Values,0,PASS


 Product validation details

In [71]:
# ============================================
# PRODUCT REFERENTIAL INTEGRITY DETAILS
# ============================================

print("PRODUCT VALIDATION DETAILS")
print("=" * 50)

print("\nPOS → Inventory")
print("Missing Product IDs:", len(missing_pos_products))
print("IDs:", sorted(missing_pos_products))

print("\nE-Commerce → Inventory")
print("Missing Product IDs:", len(missing_ecommerce_products))
print("IDs:", sorted(missing_ecommerce_products))

# Count affected records
pos_affected = pos_clean[
    pos_clean["product_id"].isin(missing_pos_products)
]

ecommerce_affected = ecommerce_clean[
    ecommerce_clean["product_id"].isin(missing_ecommerce_products)
]

print("\nAffected Records")
print("POS records:", len(pos_affected))
print("E-Commerce records:", len(ecommerce_affected))

print("\n⚠️ Supplier → Inventory validation skipped")
print("Reason: Supplier and Inventory use different Product ID formats.")

PRODUCT VALIDATION DETAILS

POS → Inventory
Missing Product IDs: 14
IDs: ['P0101', 'P0105', 'P0139', 'P0163', 'P0173', 'P0362', 'P0376', 'P0403', 'P0429', 'P0471', 'P0584', 'P0676', 'P0728', 'P0737']

E-Commerce → Inventory
Missing Product IDs: 14
IDs: ['P0101', 'P0105', 'P0139', 'P0163', 'P0173', 'P0362', 'P0376', 'P0403', 'P0429', 'P0471', 'P0584', 'P0676', 'P0728', 'P0737']

Affected Records
POS records: 161
E-Commerce records: 81

⚠️ Supplier → Inventory validation skipped
Reason: Supplier and Inventory use different Product ID formats.


Save validation_summary.csv

In [72]:
# ============================================
# SAVE VALIDATION RESULTS
# ============================================

validation_summary.to_csv(
    "/content/validation_summary.csv",
    index=False
)

print("✅ Validation completed successfully!")
print("✅ validation_summary.csv saved.")

✅ Validation completed successfully!
✅ validation_summary.csv saved.
